<a href="https://colab.research.google.com/github/Suman18-bit/Movie-Recommendation-System-/blob/main/Movie__Recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

In [8]:
DataFrame=pd.read_csv('movies_metadata.csv', engine='python', on_bad_lines='warn')

In [9]:
useFull_df=DataFrame[[
'title','genres', 'overview', 'tagline', 'vote_average',
'vote_count','popularity',
]]

In [10]:
useFull_df.sample(5)

,title,genres,overview,tagline,vote_average,vote_count,popularity
4411,Salsa,"[{'id': 18, 'name': 'Drama'}, {'id': 10402, 'n...",Fatherless barrio Puerto Rican Rico is a menia...,The Dirtiest Dancing Of Them All!,4.8,8.0,1.659488
7133,Simple Men,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",Bitter about being double-crossed by the women...,They're good boys with bad attitude!,7.1,12.0,1.595591
458,Guilty as Sin,"[{'id': 18, 'name': 'Drama'}, {'id': 53, 'name...",Before a criminal lawyer knows what has happen...,"She's finally met her match. He's handsome, we...",5.6,21.0,5.385108
6696,Chattahoochee,"[{'id': 18, 'name': 'Drama'}]","In 1955 Florida, a Korean vet has a breakdown ...",NaN,6.1,10.0,1.777657
1890,The Exorcist III,"[{'id': 53, 'name': 'Thriller'}, {'id': 27, 'n...","Set fifteen years after the original film, The...",Do you dare walk these steps again?,6.1,135.0,5.861032


In [11]:
useFull_df.iloc[0]['genres']

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [12]:
useFull_df=useFull_df.drop_duplicates().reset_index(drop=True)

In [13]:
useFull_df['vote_average']=useFull_df['vote_average'].fillna(useFull_df['vote_average'].mean())

In [14]:
useFull_df.isnull().sum()

,0
title,1
genres,0
overview,25
tagline,2587
vote_average,0
vote_count,1
popularity,0


In [15]:
useFull_df['tagline']=useFull_df['tagline'].fillna(' ')
useFull_df['overview']=useFull_df['overview'].fillna(' ')

# Convert 'popularity' and 'vote_count' to numeric, coercing errors
useFull_df['popularity'] = pd.to_numeric(useFull_df['popularity'], errors='coerce')
useFull_df['vote_count'] = pd.to_numeric(useFull_df['vote_count'], errors='coerce')

useFull_df['popularity']=useFull_df['popularity'].fillna(useFull_df['popularity'].mean())
useFull_df['vote_count']=useFull_df['vote_count'].fillna(useFull_df['vote_count'].mean())

In [16]:
useFull_df=useFull_df.dropna(subset=['title'])

In [17]:
useFull_df.isnull().sum()

,0
title,0
genres,0
overview,0
tagline,0
vote_average,0
vote_count,0
popularity,0


In [18]:
useFull_df.iloc[0]['genres']

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [19]:
import ast


In [20]:
useFull_df['genres'] = useFull_df['genres'].apply(lambda x: ' '.join(x) if isinstance(x, list) else x)

In [21]:
useFull_df.head()

,title,genres,overview,tagline,vote_average,vote_count,popularity
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ...",,7.7,5415.0,21.946943
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",When siblings Judy and Peter discover an encha...,Roll the dice and unleash the excitement!,6.9,2413.0,17.015539
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,6.5,92.0,11.712900
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,6.1,34.0,3.859495
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",Just when George Banks has recovered from his ...,Just When His World Is Back To Normal... He's ...,5.7,173.0,8.387519


In [22]:
useFull_df.head()

,title,genres,overview,tagline,vote_average,vote_count,popularity
0,Toy Story,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","Led by Woody, Andy's toys live happily in his ...",,7.7,5415.0,21.946943
1,Jumanji,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",When siblings Judy and Peter discover an encha...,Roll the dice and unleash the excitement!,6.9,2413.0,17.015539
2,Grumpier Old Men,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,6.5,92.0,11.712900
3,Waiting to Exhale,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...","Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,6.1,34.0,3.859495
4,Father of the Bride Part II,"[{'id': 35, 'name': 'Comedy'}]",Just when George Banks has recovered from his ...,Just When His World Is Back To Normal... He's ...,5.7,173.0,8.387519


In [23]:
useFull_df['combined_features'] = useFull_df['title'] + ' ' + useFull_df['genres'] + ' ' + useFull_df['tagline']

In [24]:
df=useFull_df[['title','combined_features','vote_average','vote_count','popularity']]

In [25]:
df

,title,combined_features,vote_average,vote_count,popularity
0,Toy Story,"Toy Story [{'id': 16, 'name': 'Animation'}, {'...",7.7,5415.0,21.946943
1,Jumanji,"Jumanji [{'id': 12, 'name': 'Adventure'}, {'id...",6.9,2413.0,17.015539
2,Grumpier Old Men,"Grumpier Old Men [{'id': 10749, 'name': 'Roman...",6.5,92.0,11.712900
3,Waiting to Exhale,"Waiting to Exhale [{'id': 35, 'name': 'Comedy'...",6.1,34.0,3.859495
4,Father of the Bride Part II,"Father of the Bride Part II [{'id': 35, 'name'...",5.7,173.0,8.387519
...,...,...,...,...,...
8958,35 Up,"35 Up [{'id': 99, 'name': 'Documentary'}, {'id...",7.7,8.0,0.930397
8959,Days of Being Wild,"Days of Being Wild [{'id': 80, 'name': 'Crime'...",7.3,85.0,6.242263
8960,Across the Tracks,"Across the Tracks [{'id': 18, 'name': 'Drama'}]",5.5,17.0,1.437157
8961,Begotten,"Begotten [{'id': 14, 'name': 'Fantasy'}, {'id'...",5.0,64.0,2.592052


In [26]:
import string
def remove_pun(txt):
  return txt.translate(str.maketrans('','',string.punctuation))


In [27]:
df['combined_features']=df['combined_features'].apply(remove_pun)

In [29]:
def remove_numbers(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new=new+i
  return new

In [30]:
df['combined_features']=df['combined_features'].apply(remove_numbers)

In [31]:
def remove_emoji(txt):
  e=" "
  for i in txt:
    if i.isascii():
      e=e+i
  return e

In [32]:
df['combined_features']=df['combined_features'].apply(remove_emoji)

In [33]:
import nltk
import string
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab') # Added to resolve LookupError

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [34]:
stop_words=set(stopwords.words('english')) | set(string.punctuation)

In [35]:

def remove_stopwords(txt):
  word=txt.split()
  text=[]
  for i in word:
    if i not in stop_words: # Corrected to use stop_words and convert to lowercase for comparison
      text.append(i)
  return " ".join(text) # Corrected to join the filtered words

In [36]:
df['combined_features']=df['combined_features'].apply(remove_stopwords)

In [37]:
df['combined_features']=df['combined_features'].apply(lambda x:x.lower())

In [38]:
from sklearn.preprocessing import StandardScaler

In [39]:
Scale=StandardScaler()

In [40]:
df['vote_count']=Scale.fit_transform(df[['vote_count']])

In [41]:
df['popularity']=Scale.fit_transform(df[['popularity']])

In [42]:
df['vote_average']=Scale.fit_transform(df[['vote_average']])

In [43]:
from nltk.stem import WordNetLemmatizer
import re

In [44]:
rooter=WordNetLemmatizer()

In [45]:
nltk.download('wordnet')

def lemmatize_text(text):
    if isinstance(text, str):
        word_list = text.split() # Assuming text is already preprocessed into words
        lemmatized_words = [rooter.lemmatize(word) for word in word_list]
        return ' '.join(lemmatized_words)
    return text # Return as is if not a string

df['combined_features'] = df['combined_features'].apply(lemmatize_text)

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [46]:
df['title']=DataFrame['title']

In [47]:
df=df.reset_index(drop=True)

In [48]:
df

,title,combined_features,vote_average,vote_count,popularity
0,Toy Story,toy story id name animation id name comedy id ...,1.163395,9.013375,3.408252
1,Jumanji,jumanji id name adventure id name fantasy id n...,0.578044,3.832790,2.413280
2,Grumpier Old Men,grumpier old men id name romance id name comed...,0.285369,-0.172586,1.343406
3,Waiting to Exhale,waiting exhale id name comedy id name drama id...,-0.007306,-0.272677,-0.241117
4,Father of the Bride Part II,father bride part ii id name comedy just when ...,-0.299982,-0.032804,0.672469
...,...,...,...,...,...
8958,"Welcome Home, Roxy Carmichael",up id name documentary id name foreign,1.163395,-0.317546,-0.832099
8959,35 Up,day being wild id name crime id name drama id ...,0.870720,-0.184666,0.239637
8960,Days of Being Wild,across track id name drama,-0.446319,-0.302014,-0.729854
8961,Across the Tracks,begotten id name fantasy id name horror,-0.812164,-0.220906,-0.496839


In [49]:
Indices=pd.Series(useFull_df.index,index=useFull_df['title'])

In [50]:
Tfidf=TfidfVectorizer(ngram_range=(1,3))

In [51]:
Tfidf_metrix=Tfidf.fit_transform(df['combined_features'])

In [52]:
Tfidf_metrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 308328 stored elements and shape (8963, 104666)>

In [53]:
from sklearn.metrics.pairwise import cosine_similarity

In [54]:
from nltk.corpus.reader import titles
def recommendation(title,n=7):
  if title not in Indices:
    return ["Movie not found"]
  else:
    idx=Indices[title]
    score=cosine_similarity(Tfidf_metrix[idx],Tfidf_metrix).flatten()
    similer_index=score.argsort()[::-1][0:n+1]
    return df['title'].iloc[similer_index]

In [58]:
recommendation("Jumanji")

,title
1,Jumanji
8054,The Manchurian Candidate
4854,Morgan: A Suitable Case for Treatment
1494,Head Above Water
8861,Asterix vs. Caesar
4148,Lost in America
4461,Enemies: A Love Story
581,Aladdin


In [59]:
%%writefile app.py
import streamlit as st
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# Load the necessary data components from the notebook's global state
# In a real app, you would load these from saved pickle files
@st.cache_resource
def load_data():
    # This assumes the variables exist in the environment or were saved
    # For this demo, we'll try to use the existing notebook variables via session state or re-calculation if needed
    return None

st.title('Movie Recommendation System')

# Note: Streamlit runs in a separate process. To make this work,
# we'll use the dataframes already prepared in the notebook.
# For the sake of this standalone script, it's best to save the model and data first.


Writing app.py


In [60]:
# Saving the required objects to be used by the Streamlit app
import pickle
with open('movie_data.pkl', 'wb') as f:
    pickle.dump({'df': df, 'Indices': Indices, 'Tfidf_metrix': Tfidf_metrix}, f)


In [61]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(page_title="Movie Recommender", layout="wide")

@st.cache_resource
def load_data():
    with open('movie_data.pkl', 'rb') as f:
        data = pickle.load(f)
    return data['df'], data['Indices'], data['Tfidf_metrix']

df, Indices, Tfidf_metrix = load_data()

def recommendation(title, n=7):
    if title not in Indices:
        return ["Movie not found"]
    idx = Indices[title]
    score = cosine_similarity(Tfidf_metrix[idx], Tfidf_metrix).flatten()
    similer_index = score.argsort()[::-1][1:n+1]
    return df['title'].iloc[similer_index]

st.title("🎬 Movie Recommendation System")
st.markdown("Find similar movies based on content similarity (Title, Genres, and Taglines).")

selected_movie = st.selectbox("Select a movie:", Indices.index)

if st.button('Get Recommendations'):
    res = recommendation(selected_movie)
    if isinstance(res, list):
        st.error(res[0])
    else:
        st.subheader(f"Movies similar to {selected_movie}:")
        for i, movie in enumerate(res, 1):
            st.write(f"{i}. {movie}")


Overwriting app.py


In [66]:
!pip install -q streamlit
!npm install -q -g localtunnel
import urllib
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 895ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧Password/Endpoint IP for localtunnel is: 34.21.125.60
⠙⠹

⠸⠼⠴⠦⠧your url is: https://fuzzy-wasps-allow.loca.lt
2026-06-21 05:47:47.670 Uvicorn server started on 0.0.0.0:8503

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8503
  Network URL: http://172.28.0.12:8503
  External URL: http://34.21.125.60:8503

  Stopping...
^C
